# FloodScout: train the susceptibility model

Answers one question: **given that a large flood happens in a county, which places inside it go under water?** The hazard model says *when*. This model is judged only on the *where*.

**What it learns from.** Flooded and dry pixels from the Global Flood Database, at 250 m. They are drawn in advance by `sample_susceptibility`, which reads Earth Engine, and arrive in the bundle with their SHA-256.

**The rules.** `riskengine/ml/SUSCEPTIBILITY_PROTOCOL.md` fixed the sample, features, baselines, candidates and decision rule before any data was sampled. Nothing that affects the result is set in this notebook. It comes from the config in the bundle.

**The UNOSAT test is not touched here.** It is scored once, after the decision.

## 1. Match the app's scikit-learn

The saved model is a pickle, so it must be written by the version the app runs.

In [ ]:
%pip install -q "scikit-learn==1.9.1"
import sklearn
print(sklearn.__version__)

## 2. Mount Google Drive and unpack the bundle

The notebook unpacks the newest `colab_bundle_<commit>.tar.gz` in `MyDrive/FloodScout/data/`, and checks the sample against the bundle and the config.

In [ ]:
import datetime
import hashlib
import os
import shutil
import sys
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FloodScout")
DATA_DIR = DRIVE_ROOT / "data"
RUNS_DIR = DRIVE_ROOT / "runs"
CODE_DIR = Path("/content/floodscout")

bundles = sorted(DATA_DIR.glob("colab_bundle_*.tar.gz"), key=lambda path: path.stat().st_mtime)
assert bundles, f"Put colab_bundle_<commit>.tar.gz in {DATA_DIR}"
BUNDLE = bundles[-1]

shutil.rmtree(CODE_DIR, ignore_errors=True)
with tarfile.open(BUNDLE) as bundle:
    bundle.extractall(CODE_DIR, filter="data")
os.chdir(CODE_DIR)
sys.path.insert(0, ".")

from riskengine.ml.pipeline import config as run_config

CONFIG = run_config.load(Path(Path("CONFIG").read_text().strip()))
assert CONFIG.model == "susceptibility", f"This bundle is for the {CONFIG.model} model"
SAMPLE = Path(CONFIG.value("data", "training_set"))
SAMPLE_SHA256 = hashlib.sha256(SAMPLE.read_bytes()).hexdigest()
assert SAMPLE_SHA256 == Path("DATASET_SHA256").read_text().strip(), "The sample does not match the bundle's record"
assert SAMPLE_SHA256 == CONFIG.value("data", "sha256"), "The sample does not match the config"

COMMIT = Path("COMMIT").read_text().strip()
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%MZ") + "_" + COMMIT[:7]
os.environ["FLOODSCOUT_RUN_ID"] = RUN_ID
print("Bundle", BUNDLE.name, "| run", RUN_ID)

## 3. Train

Scores the baselines, then the eight candidates with one event left out at a time, then the chosen one on spatial blocks. The decision rule then runs as written. Expect this to take over an hour. Keep the tab open.

In [ ]:
import datetime
import time

from riskengine.ml.pipeline import susceptibility

started = time.perf_counter()
# Each candidate is kept in Drive as it finishes, so a crash later on loses nothing.
CHECKPOINTS = RUNS_DIR / "checkpoints" / SAMPLE_SHA256[:12]
layer, metrics = susceptibility.run(CONFIG, RUN_ID, checkpoints=CHECKPOINTS)
RUN_DIR = susceptibility.write(RUN_ID, CONFIG, layer, metrics, RUNS_DIR)
(RUN_DIR / "ledger_line.md").write_text(
    susceptibility.ledger_line(RUN_ID, CONFIG, metrics, datetime.date.today().isoformat()) + "\n"
)
TRAIN_SECONDS = round(time.perf_counter() - started, 1)
print(f"Took {TRAIN_SECONDS} seconds. Wrote {RUN_DIR}")
print((RUN_DIR / "summary.md").read_text())

## 4. Against the baselines

The grey bars have no fitted parameters. The model must beat the best of them by 0.03 on event CV, and not fall below it on spatial blocks. The current engine scored 0.461 at sub-county level, on the test, so that figure is compared after the one UNOSAT look.

In [ ]:
import matplotlib.pyplot as plt

names = list(metrics["baselines"]) + ["model, event CV", "model, block CV"]
scores = [metrics["baselines"][name]["within_county_roc_auc"] for name in metrics["baselines"]]
scores += [metrics["event_cv"]["within_county_roc_auc"], metrics["block_cv"]["within_county_roc_auc"]]
colours = ["#9aa0a6"] * len(metrics["baselines"]) + ["#1a73e8", "#1a73e8"]

figure, axis = plt.subplots(figsize=(8, 3.5))
axis.barh(names, scores, color=colours)
axis.axvline(0.5, color="black", linewidth=1, linestyle="--")
axis.set_xlim(0.4, 1)
axis.set_xlabel("Within-county ROC AUC")
for row, score in enumerate(scores):
    axis.text(score + 0.005, row, f"{score:.3f}", va="center")
figure.tight_layout()
figure.savefig(RUN_DIR / "baselines.png", dpi=150)
plt.show()

print("Decision:", metrics["decision"])
for name, result in metrics["candidates"].items():
    print(f"{name}: {result['mean']} (sd {result['sd']})")

## 5. Every event, not only the average

A pooled score can hide an event the model gets badly wrong.

In [ ]:
best = metrics["best_baseline"]
print(f"{'event':>6} {'county-events':>14} {'model':>7} {best:>18}")
for event, row in metrics["event_cv"]["per_event"].items():
    baseline = metrics["baselines"][best]["per_event"][event]["roc_auc"]
    print(f"{event:>6} {row['county_events']:>14} {row['roc_auc']:>7} {baseline:>18}")

## 6. Calibration

Each bin holds the same weight of pixels. The closer to the diagonal, the more the probabilities can be taken as probabilities.

In [ ]:
table = metrics["calibration"]["reliability"]
figure, axis = plt.subplots(figsize=(4.5, 4.5))
axis.plot([row["mean_predicted"] for row in table], [row["observed"] for row in table], "o-")
top = max(max(row["mean_predicted"], row["observed"]) for row in table)
axis.plot([0, top], [0, top], color="black", linewidth=1, linestyle="--")
axis.set_xlabel("Predicted")
axis.set_ylabel("Observed")
figure.tight_layout()
figure.savefig(RUN_DIR / "reliability.png", dpi=150)
plt.show()
print("Brier", metrics["calibration"]["brier"])

## 7. Does it make physical sense?

Flood chance must fall with height above drainage and distance from a river, and rise with wetness. A sign the wrong way round stops the shipment until it is explained.

In [ ]:
checks = metrics["partial_dependence"]
figure, axes = plt.subplots(1, len(checks), figsize=(11, 3.2))
for axis, (feature, curve) in zip(axes, checks.items()):
    axis.plot(curve["grid"], curve["average"])
    axis.set_xlabel(feature)
    axis.set_title("as expected" if curve["as_expected"] else "WRONG WAY ROUND")
axes[0].set_ylabel("Chance of flooding")
figure.tight_layout()
figure.savefig(RUN_DIR / "partial_dependence.png", dpi=150)
plt.show()

for feature, value in sorted(metrics["permutation_importance"].items(), key=lambda item: -item[1]["mean"]):
    print(f"{feature:>16}  {value['mean']:+.4f} (sd {value['sd']})")

## 8. Bring it back

The run folder is in `MyDrive/FloodScout/runs/`. Download it with this executed notebook. A finished run is not the shipped layer until the sign check has passed and the one UNOSAT look is recorded in `riskengine/ml/experiments.md`.